# LangChain Memory 시리즈 4/7: ConversationEntityMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 엔티티(Entity) 기반 메모리가 왜 필요한지 설명할 수 있어요
- LLM이 대화에서 인물·장소·조직 등 엔티티를 자동 추출하는 방식을 이해할 수 있어요
- `entity_store.store`에서 추출된 엔티티 정보를 조회할 수 있어요
- 단순 버퍼 메모리 대비 엔티티 메모리의 장점을 설명할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| 엔티티 (Entity) | 이름이 있는 실세계의 객체 (인물, 장소, 조직 등) |
| NER (Named Entity Recognition, 개체명 인식) | 텍스트에서 엔티티를 자동으로 찾아내는 기법 |
| ConversationBufferMemory | 전체 대화를 순서대로 저장하는 기본 메모리 (1번 노트북) |

## 왜 엔티티 메모리가 필요할까요?

버퍼 메모리는 대화 전체를 저장해요. 대화가 길어지면 토큰을 많이 사용하고, 특정 인물에 대한 정보를 빠르게 찾기도 어려워요.

`ConversationEntityMemory`는 LLM을 활용해 대화에서 **엔티티를 추출**하고, 엔티티별로 정보를 구조화해서 저장해요. "민수"에 대한 정보가 궁금하면 전체 대화를 다시 읽을 필요 없이 `entity_store["민수"]`만 조회하면 돼요.

> 🎯 **강의 포인트**: 버퍼 메모리는 "언제" 말했는지가 기준, 엔티티 메모리는 "누구/무엇에 대해" 말했는지가 기준입니다. 고객 관리, CRM 시스템에서 특히 유용한 패턴임을 강조하세요.

> ⚠️ **자주 하는 실수**: `load_memory_variables({})` 호출 시 빈 딕셔너리를 전달하면 `KeyError`가 발생합니다. 반드시 `{"input": "dummy"}` 형태로 호출해야 합니다.

```mermaid
flowchart LR
    CONV["대화: 민수는 백엔드 개발자이고<br/>파이썬을 전문으로 해요"]
    CONV --> LLM_EXT[LLM 엔티티 추출]
    LLM_EXT --> STORE["entity_store<br/>{'민수': '백엔드 개발자, 파이썬 전문'}"]

    CONV2["대화: 민수는 테크스타트에서 일해요"]
    CONV2 --> LLM_EXT2[LLM 엔티티 추출]
    LLM_EXT2 --> STORE2["entity_store 업데이트<br/>{'민수': '백엔드 개발자, 파이썬 전문,<br/>테크스타트 소속'}"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    class CONV,CONV2 input
    class LLM_EXT,LLM_EXT2 process
    class STORE,STORE2 storage
```

> **참고**: 이 노트북은 레거시 메모리 클래스를 사용해요. LangChain 1.0.x에서는 엔티티 추출 도구 + `Store` 패턴이 권장돼요. 최신 방식은 14번 노트북을 참고하세요.

## 엔티티(Entity)란 무엇인가요?

프로그래밍을 처음 배우는 분들에게 "엔티티"라는 단어가 낯설 수 있어요. 아주 간단하게 설명하면:

> **엔티티(Entity) = 이름이 있는 것**

우리가 대화에서 언급하는 **사람, 장소, 조직, 제품** 등 고유한 이름을 가진 대상이 바로 엔티티예요.

| 엔티티 유형 | 예시 | 설명 |
|:---:|:---:|:---|
| 인물 | "민수", "지영" | 대화에 등장하는 사람 |
| 장소 | "서울", "강남역" | 언급되는 위치 |
| 조직 | "삼성전자", "테크스타트" | 회사, 팀, 기관 |
| 기술 | "Python", "React" | 프로그래밍 언어, 프레임워크 |

### BufferMemory vs EntityMemory: 일기장 vs 연락처 앱

두 메모리의 차이를 일상적인 비유로 이해해 볼게요.

**BufferMemory = 일기장 (日記帳)**
- 대화를 **시간 순서대로** 그대로 저장해요
- "민수는 누구야?"라고 물으면 → 일기장을 **처음부터 끝까지** 넘겨가며 민수 관련 내용을 찾아야 해요
- 대화가 100줄이면 100줄을 전부 읽어야 해요

**EntityMemory = 연락처 앱 (連絡處 App)**
- 대화에서 등장하는 인물/조직별로 **정보를 정리**해서 저장해요
- "민수는 누구야?"라고 물으면 → 연락처에서 **"민수" 검색** → 바로 정보가 나와요
- 대화가 1000줄이어도 "민수" 항목만 보면 돼요

```
[ BufferMemory - 일기장 방식 ]
───────────────────────────────
1번째 대화: "민수는 개발자야"
2번째 대화: "오늘 날씨 좋다"
3번째 대화: "지영은 디자이너야"
4번째 대화: "민수는 파이썬을 잘해"
...
→ "민수는 누구야?" → 전체를 처음부터 다 읽어야 함 😰

[ EntityMemory - 연락처 앱 방식 ]
───────────────────────────────
📇 민수: 개발자, 파이썬 전문
📇 지영: 디자이너
→ "민수는 누구야?" → "민수" 항목만 바로 조회! 😊
```

이제 실제 코드로 이 차이를 확인해 볼게요!

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 엔티티 메모리 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
# ⚠️ LangChain 1.0.x에서는 더 이상 `langchain.memory` 패키지를 사용하지 않습니다.
#    구식 메모리 클래스를 시연하기 위해 `langchain.memory`에서 불러옵니다.
from langchain.memory import ConversationEntityMemory
# ChatOpenAI: 엔티티 추출에 사용되는 LLM 인스턴스
from langchain_openai import ChatOpenAI

load_dotenv()

## 1. 기본 사용법 - 엔티티 정보 추출 및 저장

`ConversationEntityMemory`는 LLM을 사용하여 대화에서 엔티티 정보를 추출합니다.


### 💡 엔티티 메모리 초기화

```mermaid
graph TD
    LOAD_ENV["load_dotenv()"] --> LLM_INIT["ChatOpenAI(model='gpt-4o-mini', T=0)"]
    LLM_INIT --> MEM_CREATE["ConversationEntityMemory(llm=llm)"]
    MEM_CREATE --> READY[엔티티 추출 준비 완료]
```



In [ ]:
# ============================================================
# TODO: ConversationEntityMemory를 생성하세요
# 힌트: ChatOpenAI(model="gpt-4o-mini", temperature=0) → ConversationEntityMemory(llm=llm)
# 예상 결과: save_context() 호출 시 LLM이 자동으로 엔티티를 추출해 entity_store에 저장
# ============================================================

# 1단계: LLM 생성 (엔티티 추출에 사용)
# - temperature=0: 엔티티 추출 결과의 일관성 확보
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: ConversationEntityMemory 생성
# - llm 파라미터: 대화에서 엔티티와 속성을 인식하는 LLM (필수)


### 💡 대화 저장 → 엔티티 추출 흐름

```mermaid
flowchart TD
    H1[사용자 발화] --> SAVE1["save_context()"]
    SAVE1 --> HIST1[history 저장]
    HIST1 --> CALL1[LLM 호출로 엔티티 추출]
    CALL1 --> STORE1[entity_store.store 업데이트]
    STORE1 --> QUERY1[엔티티별 정보 누적]
```



### 대화 저장 및 엔티티 추출

대화를 저장하면 엔티티 정보가 자동으로 추출되어 저장됩니다.


In [ ]:
# ============================================================
# TODO: 엔티티 정보가 포함된 대화를 save_context()로 저장하세요
# 힌트: load_memory_variables() → save_context() 순서로 호출
# 예상 결과: memory.entity_store.store에 민수, 지영 등의 엔티티가 딕셔너리로 저장됨
# ============================================================

# 엔티티 정보(인물, 관계, 기술 스택)가 포함된 대화 저장
# ⚠️ ConversationEntityMemory의 내부 동작 순서:
#   1) load_memory_variables({"input": 사용자_입력}) → LLM이 엔티티명을 추출해 entity_cache에 저장
#   2) save_context() → entity_cache에 있는 엔티티를 LLM으로 요약해 entity_store에 기록
#   ConversationChain.predict()는 이 두 단계를 자동으로 처리하지만,
#   직접 호출할 때는 반드시 load_memory_variables()를 먼저 호출해야 합니다.


### 저장된 엔티티 확인

`entity_store.store`에서 추출된 엔티티 정보를 확인할 수 있습니다.


### 💡 엔티티 정보 누적 구조

```mermaid
flowchart LR
    subgraph Updates[추가 대화 입력]
        H2[새 메시지] --> SAVE2["save_context()"]
        SAVE2 --> HIST2[history 누적]
        HIST2 --> LLM2[LLM 엔티티 추출]
    end

    LLM2 --> STORE2[entity_store.store]
    STORE2 --> VIEW2[엔티티별 요약 출력]
    STORE2 --> MERGE2[기존 엔티티 정보와 병합]
```



In [ ]:
# ---------------------------------------------------
# entity_store.store에서 추출된 엔티티 조회
# ---------------------------------------------------
# entity_store.store: 엔티티 이름을 키, 요약 정보를 값으로 갖는 딕셔너리
print("=" * 60)
print("📝 추출된 엔티티 정보")
print("=" * 60)
# ⚠️ memory.entity_store는 InMemoryEntityStore 객체이므로
#    내부 딕셔너리에 접근하려면 .store 속성을 사용해야 함
entity_store = memory.entity_store.store

if entity_store:
    for entity, info in entity_store.items():
        print(f"\n🔹 {entity}:")
        print(f"   {info}")
else:
    print("아직 엔티티가 추출되지 않았습니다.")

### 💡 메모리 변수 로드 구조

```mermaid
flowchart TD
    HIST3[history]
    STORE3[entity_store.store]

    QUERY3["load_memory_variables({})"]
    HIST3 -.-> QUERY3
    STORE3 -.-> QUERY3

    QUERY3 --> OUTPUT_ENT["'entities' 키"]
    QUERY3 --> OUTPUT_HIST["'history' 키"]
    OUTPUT_ENT --> PRINT_ENT[엔티티 요약 출력]
    OUTPUT_HIST --> PRINT_HIST[대화 기록 출력]
```



### 추가 대화로 엔티티 정보 축적

더 많은 대화를 저장하면 엔티티 정보가 점진적으로 축적됩니다.


### 💡 프로젝트 관리 시나리오 흐름

```mermaid
flowchart TD
    subgraph ProjectChat
        STEP1[팀 구성 정보 입력] --> SAVE_P1[save_context]
        STEP2[프로젝트 정보 입력] --> SAVE_P2[save_context]
        STEP3[역할 분담 입력] --> SAVE_P3[save_context]
        STEP4[팀장 역할 입력] --> SAVE_P4[save_context]
    end

    SAVE_P1 & SAVE_P2 & SAVE_P3 & SAVE_P4 --> LLM_P[엔티티 추출 LLM]
    LLM_P --> STORE_P[project_memory.entity_store]
    STORE_P --> VIEW_P[엔티티별 정보 확인]
```



In [ ]:
# ============================================================
# TODO: 추가 대화 2개를 저장하여 엔티티 정보를 업데이트하세요
# 힌트: load_memory_variables() → save_context()를 2회 반복
# 예상 결과: 민수/지영의 기술 스택과 "테크스타트" 엔티티가 추가/업데이트됨
# ============================================================

# 1단계: 기술 스택 정보 추가 (기존 엔티티에 새 속성이 병합됨)


# 2단계: 스타트업 이름 정보 추가


# 업데이트된 엔티티 정보 확인


## 2. 메모리 변수 로드

`load_memory_variables()`를 사용하여 저장된 엔티티 정보와 대화 기록을 확인할 수 있습니다.


### 💡 엔티티 정보를 활용한 응답 구성

```mermaid
flowchart LR
    STORE_P2[project_memory.entity_store] --> LOAD_P["load_memory_variables({})"]
    HIST_P[project_memory.history] --> LOAD_P
    LOAD_P --> ENT_OUT[entities 문자열]
    LOAD_P --> HIST_OUT[history 문자열]
    ENT_OUT --> PROMPT_P[프롬프트에 엔티티 정보 삽입]
    HIST_OUT --> PROMPT_P
    PROMPT_P --> ANSWER_P[맥락 유지 응답 생성]
```



### Before vs After: "민수는 누구야?"라고 물었을 때

위의 `load_memory_variables()` 결과를 보면 EntityMemory의 강점이 명확해요.
같은 질문에 대해 BufferMemory와 EntityMemory가 어떻게 다르게 동작하는지 비교해 볼게요.

#### Before: BufferMemory 방식

```
질문: "민수는 누구야?"

→ 전체 대화 이력을 순서대로 검색해야 함:
  Human: 민수와 지영은 같은 팀에서 일하는 동료예요...
  AI: 흥미로운 정보네요!...
  Human: 민수는 파이썬과 Django를 전문으로 하고...
  AI: 좋은 기술 스택이네요!...
  Human: 네, 맞아요. 그리고 그들의 스타트업 이름은...
  AI: 테크스타트라는 이름이 멋지네요!...

→ 대화 6줄을 전부 프롬프트에 넣어야 함 (토큰 낭비!)
→ 대화가 100줄이면? 100줄 전부 프롬프트에 포함...
```

#### After: EntityMemory 방식

```
질문: "민수는 누구야?"

→ entity_store["민수"]를 바로 조회:
  "민수는 백엔드 개발자로, 파이썬과 Django를 전문으로 하며,
   지영과 함께 스타트업을 창업하기로 결정한 동료입니다."

→ 딱 1줄의 요약 정보만 프롬프트에 넣으면 됨!
→ 대화가 1000줄이어도 요약 정보는 1줄!
```

#### 한눈에 비교

| 비교 항목 | BufferMemory | EntityMemory |
|:---:|:---:|:---:|
| 검색 방식 | 전체 대화 순차 탐색 | 엔티티 이름으로 직접 조회 |
| "민수" 질문 시 | 대화 전체에서 민수 관련 부분 찾기 | `entity_store["민수"]` 바로 반환 |
| 토큰 사용량 | 대화 길이에 비례하여 증가 | 엔티티 요약 크기만큼만 사용 |
| 정보 형태 | 원본 대화 텍스트 그대로 | LLM이 요약한 구조화된 정보 |
| 적합한 상황 | 짧은 대화, 순서가 중요한 경우 | 고객 관리(CRM), 인물 정보 추적 |

In [ ]:
# ============================================================
# TODO: load_memory_variables()로 entities와 history를 동시 조회하세요
# 힌트: memory.load_memory_variables({"input": "민수와 지영에 대해 알려주세요"})
# 예상 결과: 'entities' 키에 엔티티 요약, 'history' 키에 대화 기록이 반환됨
# ============================================================

# ⚠️ load_memory_variables({"input": 질문})는 질문 텍스트에서 엔티티명을 추출한 뒤
#    entity_store에서 해당 엔티티의 정보를 조회합니다.
#    "dummy"처럼 엔티티명이 없는 입력을 넣으면 entities가 빈 딕셔너리로 반환됩니다.

# 여기에 코드를 작성하세요
pass

## 3. 실전 예제: 프로젝트 관리 챗봇

프로젝트 관리 시나리오에서 팀원, 프로젝트, 작업 등의 엔티티 정보를 추적하는 예제입니다.


### 💡 ConversationChain을 활용한 엔티티 메모리 채팅

앞에서는 `load_memory_variables()` → `save_context()`를 직접 호출했어요.
실전에서는 `ConversationChain`이 이 두 단계를 **자동으로 처리**해 줍니다.

```mermaid
flowchart LR
    USER[사용자 입력] --> CHAIN["conversation.predict(input=...)"]
    CHAIN --> STEP1["① load_memory_variables()<br/>엔티티명 추출 → entity_cache"]
    STEP1 --> STEP2["② LLM 호출<br/>엔티티 정보 + 대화 이력을 프롬프트에 주입"]
    STEP2 --> STEP3["③ save_context()<br/>entity_cache 기반으로 entity_store 업데이트"]
    STEP3 --> RESPONSE[AI 응답 반환]
```

`ENTITY_MEMORY_CONVERSATION_TEMPLATE`는 LangChain이 제공하는 엔티티 메모리 전용 프롬프트로,
`{entities}`와 `{history}` 변수를 프롬프트에 자동 삽입합니다.

## 4. 엔티티 정보 활용

저장된 엔티티 정보를 활용하여 대화 맥락을 유지할 수 있습니다.


In [ ]:
# ---------------------------------------------------
# 엔티티 정보를 포함한 메모리 변수 로드 및 활용
# ---------------------------------------------------
# ⚠️ 레거시 버그 우회: input_key 포함 필수
# memory_vars = memory.load_memory_variables({})  # KeyError 발생!
memory_vars = memory.load_memory_variables({"input": "민수와 지영에 대해 알려주세요"})

print("=" * 60)
print("📊 엔티티 기반 메모리 활용")
print("=" * 60)

# "entities" 키: 현재 입력("dummy")과 관련된 엔티티 요약 텍스트
# 체인 프롬프트의 {entities} 변수에 주입하면 LLM이 인물·조직 정보를 인식한 채 응답
if "entities" in memory_vars:
    print("\n💡 저장된 엔티티 정보:")
    print(memory_vars["entities"])
    print("\n이 정보를 활용하여 대화 맥락을 유지하고,")
    print("엔티티에 대한 질문에 정확하게 답변할 수 있습니다.")

## 핵심 정리

```python
# ConversationEntityMemory 기본 사용법
from langchain.memory import ConversationEntityMemory
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationEntityMemory(llm=llm)

# 대화 저장 (엔티티 자동 추출)
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 엔티티 정보 확인
entities = memory.entity_store.store

# 메모리 변수 로드
memory_vars = memory.load_memory_variables({})
```

### 주요 특징
- ✅ **엔티티 추출**: LLM을 사용하여 대화에서 엔티티 정보 자동 추출
- ✅ **지식 축적**: 시간이 지나면서 엔티티 정보를 점진적으로 축적
- ✅ **구조화된 저장**: 엔티티별로 정보를 구조화하여 저장
- ✅ **효율적 메모리**: 엔티티 정보를 별도로 관리하여 효율적인 메모리 사용

### 추출 가능한 엔티티 유형
- 👤 **인물**: 이름, 역할, 관계 등
- 🏢 **조직**: 회사명, 부서, 프로젝트 등
- 📍 **장소**: 위치, 주소 등
- 📅 **날짜/시간**: 일정, 기간 등
- 🎯 **기타**: 제품, 서비스, 개념 등

### 주의사항
- ⚠️ **LLM 필요**: 엔티티 추출을 위해 LLM 인스턴스가 필요함
- ⚠️ **추출 정확도**: LLM의 성능에 따라 엔티티 추출 정확도가 달라짐
- ⚠️ **비용**: 엔티티 추출에 LLM 호출이 필요하므로 비용 발생
- ⚠️ **대화형 사용**: ConversationChain과 함께 사용하는 것이 일반적 (deprecated 경고 있음)

In [ ]:
# ---------------------------------------------------
# ConversationChain + EntityMemory로 실전 채팅 구성
# ---------------------------------------------------
from langchain.chains import ConversationChain
from langchain.memory import ConversationEntityMemory
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE

# 엔티티 메모리 전용 프롬프트 확인
print("📋 엔티티 메모리 프롬프트 템플릿:")
print("=" * 60)
print(ENTITY_MEMORY_CONVERSATION_TEMPLATE.template)
print("=" * 60)

In [ ]:
# ---------------------------------------------------
# ConversationChain 생성 (엔티티 메모리 연동)
# ---------------------------------------------------
# ConversationChain.predict()는 내부적으로:
#   1) load_memory_variables() → 엔티티 추출 + entity_cache 저장
#   2) LLM 호출 (엔티티 정보 + 대화 이력이 프롬프트에 자동 주입)
#   3) save_context() → entity_cache 기반 entity_store 업데이트
# 를 자동으로 처리합니다.

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

conversation = ConversationChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=ConversationEntityMemory(llm=llm),
)

In [ ]:
# ============================================================
# TODO: conversation.predict()로 팀 구성 정보를 입력하세요
# 힌트: conversation.predict(input="우리 프로젝트 팀을 소개할게요. 팀장은 김서연이고...")
# 예상 결과: AI가 팀 구성에 대해 응답하고, entity_store에 김서연/박준호/이하은이 저장됨
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ---------------------------------------------------
# 1차 대화 후 entity_store 확인
# ---------------------------------------------------
# ConversationChain.predict()가 자동으로 엔티티를 추출하여 store에 저장했는지 확인
print("📝 1차 대화 후 엔티티:")
print("-" * 40)
for entity, info in conversation.memory.entity_store.store.items():
    print(f"🔹 {entity}: {info}")

In [ ]:
# ============================================================
# TODO: conversation.predict()로 프로젝트 정보를 추가하세요
# 힌트: conversation.predict(input="우리 프로젝트 이름은 'SmartEdu'이고...")
# 예상 결과: SmartEdu 엔티티가 추가되고, 박준호/이하은의 역할이 업데이트됨
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ============================================================
# TODO: conversation.predict()로 엔티티 기반 맥락 유지를 확인하세요
# 힌트: conversation.predict(input="박준호가 요즘 FastAPI를 공부하고 있대요...")
# 예상 결과: AI가 박준호의 기존 정보(3년차 백엔드, API 담당)를 기억하며 응답
# ============================================================

# 여기에 코드를 작성하세요
pass

In [ ]:
# ---------------------------------------------------
# 최종 entity_store 확인: 대화를 통해 축적된 엔티티 정보
# ---------------------------------------------------
print("=" * 60)
print("📚 최종 엔티티 정보 (3회 대화 후)")
print("=" * 60)
for entity, info in conversation.memory.entity_store.store.items():
    print(f"\n🔹 {entity}:")
    print(f"   {info}")